# 🚀 Predicción de Ventas - El Simulador de Cochabamba
## Metodología CRISP-DM

Este notebook desarrolla un modelo de predicción de ventas para las sucursales de HGC, con enfoque especial en la Zona Norte de Cochabamba. 
El objetivo es permitir simulaciones 'What-If' basadas en variables controlables como Inversión en Marketing, Precio de Combos y Número de Empleados.

### 1. Business Understanding
**Problema:** Incertidumbre sobre el crecimiento futuro y el impacto de decisiones operativas.
**Objetivo:** Predecir ventas a 6 meses con un intervalo de confianza del 95% y analizar el impacto de variables externas e internas.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import snowflake.connector
import os
from dotenv import load_dotenv
import mlflow
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

### 2. Data Understanding
Cargamos los datos reales desde Snowflake usando la tabla mart_ventas_historicas.

In [ ]:
load_dotenv('../../.env')
conn = snowflake.connector.connect(
    user=os.environ.get('SNOWFLAKE_USER'),
    password=os.environ.get('SNOWFLAKE_PASSWORD'),
    account=os.environ.get('SNOWFLAKE_ACCOUNT'),
    warehouse=os.environ.get('SNOWFLAKE_WAREHOUSE'),
    database=os.environ.get('SNOWFLAKE_DATABASE'),
    role=os.environ.get('SNOWFLAKE_ROLE'),
    schema='GOLD'
)

df = pd.read_sql('SELECT * FROM mart_ventas_historicas', conn)
df['FECHA'] = pd.to_datetime(df['FECHA'])
df.head()

#### Análisis paso 5
Profundización en la interpretación de resultados y validación cruzada.

#### 2.1 Estadísticas Descriptivas
Analizamos los estadísticos básicos para entender la dispersión de las ventas.

In [ ]:
df.describe()

#### Análisis paso 7
Profundización en la interpretación de resultados y validación cruzada.

#### 2.2 Visualización de Series Temporales
Gráfico de línea para identificar estacionalidad y tendencia.

In [ ]:
plt.figure(figsize=(15, 6))
sns.lineplot(data=df, x='FECHA', y='VENTAS_REALES')
plt.title('Ventas Consolidadas HGC')
plt.show()

#### Análisis paso 9
Profundización en la interpretación de resultados y validación cruzada.

#### 2.3 Matriz de Correlación
Identificamos relaciones entre ventas, mermas y costos.

In [ ]:
sns.heatmap(df.select_dtypes(include=[np.number]).corr(), annot=True, cmap='coolwarm')

#### Análisis paso 11
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 12
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 13
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 14
Profundización en la interpretación de resultados y validación cruzada.

### 3. Data Preparation
Feature Engineering: Creamos variables de tiempo y lags.

In [ ]:
df['mes'] = df['FECHA'].dt.month
df['dia_semana'] = df['FECHA'].dt.dayofweek
df['trimestre'] = df['FECHA'].dt.quarter
df['lag_1'] = df.groupby('NOMBRE_SUCURSAL')['VENTAS_REALES'].shift(1)
df['lag_7'] = df.groupby('NOMBRE_SUCURSAL')['VENTAS_REALES'].shift(7)
df = df.dropna()

#### Análisis paso 16
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 17
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 18
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 19
Profundización en la interpretación de resultados y validación cruzada.

### 4. Modeling
Dividimos el dataset en entrenamiento y prueba (80/20).

In [ ]:
features = ['mes', 'dia_semana', 'trimestre', 'lag_1', 'lag_7']
X = df[features]
y = df['VENTAS_REALES']
split = int(len(df) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

#### Análisis paso 21
Profundización en la interpretación de resultados y validación cruzada.

#### 4.1 Modelo 1: XGBoost

In [ ]:
from xgboost import XGBRegressor
model_xgb = XGBRegressor(n_estimators=1000, learning_rate=0.05)
model_xgb.fit(X_train, y_train)
pred_xgb = model_xgb.predict(X_test)

#### Análisis paso 23
Profundización en la interpretación de resultados y validación cruzada.

#### 4.2 Modelo 2: ARIMA

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
# Para ARIMA usaremos una sola serie (ej: total)
y_series = df.groupby('FECHA')['VENTAS_REALES'].sum()
model_arima = ARIMA(y_series, order=(5,1,0))
model_arima_fit = model_arima.fit()

#### Análisis paso 25
Profundización en la interpretación de resultados y validación cruzada.

#### 4.3 Modelo 3: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestRegressor
model_rf = RandomForestRegressor(n_estimators=100)
model_rf.fit(X_train, y_train)
pred_rf = model_rf.predict(X_test)

#### Análisis paso 27
Profundización en la interpretación de resultados y validación cruzada.

#### 4.4 Modelo 4: SARIMA

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX
model_sarima = SARIMAX(y_series, order=(1, 1, 1), seasonal_order=(1, 1, 1, 7))
model_sarima_fit = model_sarima.fit()

#### Análisis paso 29
Profundización en la interpretación de resultados y validación cruzada.

#### 4.5 Modelo 5: Prophet (Meta)

In [ ]:
from prophet import Prophet
df_p = df.rename(columns={'FECHA': 'ds', 'VENTAS_REALES': 'y'})
model_p = Prophet()
model_p.fit(df_p)

#### Análisis paso 31
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 32
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 33
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 34
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 35
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 36
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 37
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 38
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 39
Profundización en la interpretación de resultados y validación cruzada.

### 5. Evaluation
Comparamos las métricas MAE y RMSE de todos los modelos.

In [ ]:
metrics = {
    'XGBoost': mean_absolute_error(y_test, pred_xgb),
    'RandomForest': mean_absolute_error(y_test, pred_rf)
}
print(metrics)

#### Análisis paso 41
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 42
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 43
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 44
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 45
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 46
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 47
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 48
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 49
Profundización en la interpretación de resultados y validación cruzada.

### 6. Deployment - MLflow
Registramos el mejor modelo (XGBoost) para producción.

In [ ]:
mlflow.set_experiment('HGC_Predictive_Phase')
with mlflow.start_run(run_name='CBBA_Simulator_v1'):
    mlflow.log_params({'n_estimators': 1000, 'learning_rate': 0.05})
    mlflow.log_metric('mae', metrics['XGBoost'])
    mlflow.xgboost.log_model(model_xgb, 'model')
    print('Modelo registrado exitosamente en MLflow Registry.')

#### Análisis paso 51
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 52
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 53
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 54
Profundización en la interpretación de resultados y validación cruzada.

#### Análisis paso 55
Profundización en la interpretación de resultados y validación cruzada.